<a href="https://colab.research.google.com/github/prof9463-cloud/garment-production-risk-triage/blob/main/Garment_productivity_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the file
df = pd.read_csv("garments_worker_productivity.csv")

# Fix department names so typos don't split categories
df['clean_dept'] = df['department'].str.strip().replace({'sweing': 'sewing'})

# Check missing WIP rows by department
check = df.groupby('clean_dept')['wip'].apply(lambda group: pd.Series({
    'Total Rows': len(group),
    'Missing WIP': group.isna().sum(),
    'Missing %': (group.isna().mean() * 100).round(1)
}))

print(check)

clean_dept             
finishing   Total Rows     506.0
            Missing WIP    506.0
            Missing %      100.0
sewing      Total Rows     691.0
            Missing WIP      0.0
            Missing %        0.0
Name: wip, dtype: float64


In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. Load data and clean department text bug
df = pd.read_csv("garments_worker_productivity.csv")
df['clean_dept'] = df['department'].str.strip().replace({'sweing': 'sewing'})
df['miss'] = df['actual_productivity'] < df['targeted_productivity']

# 2. Strict chronological ordering (oldest to newest)
df['date'] = pd.to_datetime(df['date'])
df_sorted = df.sort_values(by='date').reset_index(drop=True)

# 3. Create 'yesterday_miss' feature (grouped by department + team)
df_sorted['yesterday_miss'] = df_sorted.groupby(['clean_dept', 'team'])['miss'].shift(1)
df_sorted['yesterday_miss_feat'] = df_sorted['yesterday_miss'].fillna(False).astype(int)

# 4. Feature matrix and 80/20 chronological train/test split
features = ['clean_dept', 'team', 'targeted_productivity', 'no_of_workers', 'smv', 'over_time', 'yesterday_miss_feat']
X_all = pd.get_dummies(df_sorted[features], drop_first=True)
y_all = df_sorted['miss']

split_idx = int(len(df_sorted) * 0.80)
X_train = X_all.iloc[:split_idx]
X_test  = X_all.iloc[split_idx:]
y_train = y_all.iloc[:split_idx]
y_test  = y_all.iloc[split_idx:]

# 5. Logistic Regression Model
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
lr_risk = lr.predict_proba(X_test)[:, 1]

eval_lr = X_test.copy()
eval_lr['risk_score'] = lr_risk
eval_lr['actual_miss'] = y_test
lr_top10_misses = eval_lr.sort_values(by='risk_score', ascending=False).head(10)['actual_miss'].sum()

# 6. Random Forest Model (Phase 2 Benchmark)
rf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf.fit(X_train, y_train)
rf_risk = rf.predict_proba(X_test)[:, 1]

eval_rf = X_test.copy()
eval_rf['risk_score'] = rf_risk
eval_rf['actual_miss'] = y_test
rf_top10_misses = eval_rf.sort_values(by='risk_score', ascending=False).head(10)['actual_miss'].sum()

print("=" * 55)
print("PHASE 2 MODEL COMPARISON (TEST SET: 240 SHIFTS)")
print("=" * 55)
print(f"Lazy Guess Accuracy ('Always Safe'): {(y_test == False).mean() * 100:.2f}%")
print(f"Logistic Regression Accuracy:        {lr.score(X_test, y_test) * 100:.2f}% | Precision@10: {lr_top10_misses}/10 ({lr_top10_misses * 10}%)")
print(f"Random Forest Accuracy:              {rf.score(X_test, y_test) * 100:.2f}% | Precision@10: {rf_top10_misses}/10 ({rf_top10_misses * 10}%)")

# 7. Morning Floor Triage Sheet (Phase 3)
test_display = df_sorted.iloc[split_idx:].copy()
test_display['risk_score'] = rf_risk
latest_date = test_display['date'].max()

sheet = test_display[test_display['date'] == latest_date].copy()
sheet['Risk Level'] = sheet['risk_score'].apply(lambda x: 'HIGH RISK' if x >= 0.40 else 'NORMAL')
sheet['Risk Score'] = (sheet['risk_score'] * 100).round(1).astype(str) + '%'
sheet['Yesterday Missed?'] = sheet['yesterday_miss_feat'].apply(lambda x: 'YES' if x == 1 else 'NO')

print("\n" + "=" * 55)
print(f"PHASE 3 MORNING TRIAGE SHEET ({latest_date.strftime('%Y-%m-%d')})")
print("=" * 55)
display(sheet[['team', 'clean_dept', 'targeted_productivity', 'Yesterday Missed?', 'Risk Score', 'Risk Level']].sort_values(by='Risk Score', ascending=False).reset_index(drop=True))

/tmp/ipykernel_20765/458925799.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_sorted['yesterday_miss_feat'] = df_sorted['yesterday_miss'].fillna(False).astype(int)


PHASE 2 MODEL COMPARISON (TEST SET: 240 SHIFTS)
Lazy Guess Accuracy ('Always Safe'): 75.00%
Logistic Regression Accuracy:        77.50% | Precision@10: 6/10 (60%)
Random Forest Accuracy:              80.00% | Precision@10: 9/10 (90%)

PHASE 3 MORNING TRIAGE SHEET (2015-03-11)


,team,clean_dept,targeted_productivity,Yesterday Missed?,Risk Score,Risk Level
0,8,finishing,0.70,YES,75.0%,HIGH RISK
1,6,finishing,0.70,YES,72.5%,HIGH RISK
2,10,finishing,0.75,YES,66.2%,HIGH RISK
3,7,finishing,0.65,NO,53.1%,HIGH RISK
4,7,sewing,0.65,NO,52.5%,HIGH RISK
5,8,sewing,0.70,NO,52.1%,HIGH RISK
6,2,finishing,0.75,NO,36.8%,NORMAL
7,12,finishing,0.80,NO,34.1%,NORMAL
8,11,finishing,0.80,YES,33.5%,NORMAL
9,9,finishing,0.75,NO,28.6%,NORMAL


In [ ]:
# Print min and max dates for the training and test splits
train_dates = df_sorted.iloc[:split_idx]['date']
test_dates  = df_sorted.iloc[split_idx:]['date']

print("--- DATE RANGE AUDIT ---")
print(f"Train Set (80% / {len(train_dates)} rows): {train_dates.min().strftime('%Y-%m-%d')} to {train_dates.max().strftime('%Y-%m-%d')}")
print(f"Test Set  (20% / {len(test_dates)} rows):  {test_dates.min().strftime('%Y-%m-%d')} to {test_dates.max().strftime('%Y-%m-%d')}")

print("\n--- SPLIT METHODOLOGY CONFIRMATION ---")
print("1. Did train_test_split shuffle?: NO (train_test_split was NOT used).")
print("2. Split implementation: Positional slice on date-sorted DataFrame (.iloc[:split_idx] & .iloc[split_idx:]).")
print("3. Are Train/Test sets identical between LR and RF?: YES (both models fit the exact same X_train and evaluated on X_test).")

--- DATE RANGE AUDIT ---
Train Set (80% / 957 rows): 2015-01-01 to 2015-02-26
Test Set  (20% / 240 rows):  2015-02-26 to 2015-03-11

--- SPLIT METHODOLOGY CONFIRMATION ---
1. Did train_test_split shuffle?: NO (train_test_split was NOT used).
2. Split implementation: Positional slice on date-sorted DataFrame (.iloc[:split_idx] & .iloc[split_idx:]).
3. Are Train/Test sets identical between LR and RF?: YES (both models fit the exact same X_train and evaluated on X_test).


In [ ]:
print(f"Top 10 Overlap Count: {len(overlap)} / 10 rows")
print(f"Rows unique to Random Forest's Top 10: {len(rf_only)}\n")

# 2. Inspect the unique rows caught by Random Forest
print("--- Rows in RF Top 10 but NOT in Logistic Regression Top 10 ---")
display(rf_only[['clean_dept_sewing', 'team', 'targeted_productivity', 'yesterday_miss_feat', 'risk_score', 'actual_miss']])

# 3. Check yesterday_miss_feat distribution among RF's 9 successful catches
rf_hits = rf_top10_df[rf_top10_df['actual_miss'] == True]
print("\n--- Distribution of yesterday_miss_feat among RF's 9 actual misses caught ---")


Top 10 Overlap Count: 1 / 10 rows
Rows unique to Random Forest's Top 10: 9

--- Rows in RF Top 10 but NOT in Logistic Regression Top 10 ---


,clean_dept_sewing,team,targeted_productivity,yesterday_miss_feat,risk_score,actual_miss
965,False,2,0.70,1,0.715305,True
969,False,9,0.75,1,0.745021,True
1035,False,6,0.75,1,0.710896,True
971,False,6,0.75,1,0.710896,True
1196,False,6,0.70,1,0.725171,True
986,False,9,0.75,1,0.745021,True
1180,False,8,0.70,1,0.749705,True
989,False,2,0.70,1,0.715305,True
1150,False,2,0.70,1,0.715305,False



--- Distribution of yesterday_miss_feat among RF's 9 actual misses caught ---
